## JAS PTSUV

#### Check SBE37 CTD files

In [ ]:
# Imports
import numpy as np
import xarray as xrd
import re
import matplotlib.pyplot as pltm
import matplotlib.dates as mdates
from matplotlib.colors import LogNorm
import pandas as pd
import plotly as plt
from plotly.subplots import make_subplots
import plotly.graph_objects as go


In [ ]:
SBE37_42m = '/datasets/work/oa-srsalt/work/preqa/SWOT/cal_val/jason_calval/all_mooring_data/data_in/rec_202508/BASJAS_PTSUV_202508/SBE37_9914/SBE37SM-RS232_03709914_2025_08_26.cnv'
SBE37_30m = '/datasets/work/oa-srsalt/work/preqa/SWOT/cal_val/jason_calval/all_mooring_data/data_in/rec_202508/BASJAS_PTSUV_202508/SBE37_8853/SBE37SM-RS232_03708853_2025_09_22.cnv'
SBE37_17m = '/datasets/work/oa-srsalt/work/preqa/SWOT/cal_val/jason_calval/all_mooring_data/data_in/rec_202508/BASJAS_PTSUV_202508/SBE37_8852/SBE37SM-RS232_03708852_2025_08_27.cnv'

In [ ]:
def _clean_label(var_code: str, desc: str) -> str:
    """Convert CNV header fields into readable dataframe column names."""
    v = var_code.lower().strip()
    d = desc.strip()

    # Extract unit if present, e.g. [S/m], [PSU], [db]
    unit_match = re.search(r"\[[^\]]+\]", d)
    unit = unit_match.group(0) if unit_match else ""

    # Main term before comma, e.g. 'Salinity' from 'Salinity, Practical [PSU]'
    main = d.split(",", 1)[0].strip()

    if v.startswith("time") or "julian" in d.lower():
        return "Instrument Time"
    if v.startswith("tv") or "temperature" in d.lower():
        return "Temperature"
    if v.startswith("cond") or "conductivity" in d.lower():
        return f"Conductivity {unit}".strip()
    if v.startswith("sal") or "salinity" in d.lower():
        return f"Salinity {unit}".strip()
    if v.startswith("prd") or "pressure" in d.lower():
        return f"Pressure {unit}".strip()
    if v.startswith("flag") or main.lower() == "flag":
        return "flag"

    # Generic fallback: keep cleaned main term + unit
    return f"{main} {unit}".strip()

In [ ]:
with open(SBE37_42m, "r", encoding="utf-8", errors="ignore") as f:
    lines = f.readlines()

col_names = []
start = None

for i, line in enumerate(lines):
    s = line.strip()
    if s == "*END*":
        start = i + 1

    # Example: # name 1 = tv290C: Temperature [ITS-90, deg C]
    m = re.match(r"#\s*name\s+\d+\s*=\s*([^:]+):\s*(.+)$", s)
    if m:
        var_code = m.group(1).strip()
        desc = m.group(2).strip()
        col_names.append(_clean_label(var_code, desc))

if start is None:
    raise ValueError("Could not find '*END*' in CNV header.")

SBE37_42m_df = pd.read_csv(
    SBE37_42m,
    skiprows=start,
    sep=r"\s+",
    header=None,
    names=col_names if col_names else None,
    engine="python",
)

print(SBE37_42m_df)

In [ ]:
with open(SBE37_30m, "r", encoding="utf-8", errors="ignore") as f:
    lines = f.readlines()

col_names = []
start = None

for i, line in enumerate(lines):
    s = line.strip()
    if s == "*END*":
        start = i + 1

    # Example: # name 1 = tv290C: Temperature [ITS-90, deg C]
    m = re.match(r"#\s*name\s+\d+\s*=\s*([^:]+):\s*(.+)$", s)
    if m:
        var_code = m.group(1).strip()
        desc = m.group(2).strip()
        col_names.append(_clean_label(var_code, desc))

if start is None:
    raise ValueError("Could not find '*END*' in CNV header.")

SBE37_30m_df = pd.read_csv(
    SBE37_30m,
    skiprows=start,
    sep=r"\s+",
    header=None,
    names=col_names if col_names else None,
    engine="python",
)

print(SBE37_30m_df)

In [ ]:
with open(SBE37_17m, "r", encoding="utf-8", errors="ignore") as f:
    lines = f.readlines()

col_names = []
start = None

for i, line in enumerate(lines):
    s = line.strip()
    if s == "*END*":
        start = i + 1

    # Example: # name 1 = tv290C: Temperature [ITS-90, deg C]
    m = re.match(r"#\s*name\s+\d+\s*=\s*([^:]+):\s*(.+)$", s)
    if m:
        var_code = m.group(1).strip()
        desc = m.group(2).strip()
        col_names.append(_clean_label(var_code, desc))

if start is None:
    raise ValueError("Could not find '*END*' in CNV header.")

SBE37_17m_df = pd.read_csv(
    SBE37_17m,
    skiprows=start,
    sep=r"\s+",
    header=None,
    names=col_names if col_names else None,
    engine="python",
)

print(SBE37_17m_df)

In [ ]:

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=("Temperature", "Salinity"),
)

fig.add_scatter(
    x=SBE37_42m_df["Instrument Time"],
    y=SBE37_42m_df["Temperature"],
    mode="lines",
    name="42 m",
    row=1,
    col=1,
)
fig.add_scatter(
    x=SBE37_30m_df["Instrument Time"],
    y=SBE37_30m_df["Temperature"],
    mode="lines",
    name="30 m",
    row=1,
    col=1,
)
fig.add_scatter(
    x=SBE37_17m_df["Instrument Time"],
    y=SBE37_17m_df["Temperature"],
    mode="lines",
    name="17 m",
    row=1,
    col=1,
)

fig.add_scatter(
    x=SBE37_42m_df["Instrument Time"],
    y=SBE37_42m_df["Salinity [PSU]"],
    mode="lines",
    name="42 m",
    showlegend=False,
    row=2,
    col=1,
)
fig.add_scatter(
    x=SBE37_30m_df["Instrument Time"],
    y=SBE37_30m_df["Salinity [PSU]"],
    mode="lines",
    name="30 m",
    showlegend=False,
    row=2,
    col=1,
)
fig.add_scatter(
    x=SBE37_17m_df["Instrument Time"],
    y=SBE37_17m_df["Salinity [PSU]"],
    mode="lines",
    name="17 m",
    showlegend=False,
    row=2,
    col=1,
)

fig.update_layout(
    title="SBE37 comparison",
    template="plotly_white",
    width=1400,
    height=900,
    margin=dict(t=80, b=40, l=60, r=30),
    legend_title_text="Sensor depth",
)

fig.update_xaxes(title_text="Instrument Time", row=2, col=1)
fig.update_yaxes(title_text="Temperature", row=1, col=1)
fig.update_yaxes(title_text="Salinity [PSU]", row=2, col=1)

fig.show()

#### Check SBE26